# ANÁLISIS Y ESTANDARIZACIÓN DE PRODUCTOS

In [6]:
import sys
import os
project_root = os.path.dirname(os.getcwd()) 
sys.path.insert(0, project_root)
from utils.imports import *

## PRODUCTOS ODOO - INTERFUERZA

In [7]:
notebook_dir = Path.cwd()
raw_path = notebook_dir.parent / 'data' / 'raw'
processed_path = notebook_dir.parent / 'data' / 'processed'
df_odoo = pd.read_csv(raw_path / 'ProductosOdoo.csv')
df_odoo_copy = df_odoo.copy()
df_int = pd.read_excel(raw_path / 'ProductosInterfuerza.xlsx')
df_int_copy = df_int.copy()

display(Markdown("### Productos Odoo"))
print(f"Total productos: {len(df_odoo)}")
display(df_odoo.head(5))
display(Markdown("### Productos Interfuerza"))
print(f"Total productos: {len(df_int)}")
display(df_int.head(5))

### Productos Odoo

Total productos: 14215


,id,name,barcode,default_code,x_studio_many2many_field_37q_1irl4uc58,categ_id,seller_ids
0,__export__.product_template_31022_88373f19,ADORNO HALLOWEN,7453066213137,CF00614/autoriza noris,NaN,Cumpleaños / Artículos de Fiesta,NaN
1,__export__.product_template_219315_4256fd8c,ARCHIVADOR VERTICAL 3 GAVETAS Y CERRADURA,NaN,NaN,NaN,All,NaN
2,__export__.product_template_219316_dfced89f,ARCHIVADOR VERTICAL 4 GAVETAS Y CERRADURA,NaN,NaN,NaN,All,NaN
3,__export__.product_template_76612_602c2239,ARTICULOS DE NAVIDAD,93,9,NaN,Navidad / Bolas,NaN
4,__export__.product_template_29991_44f2fe53,ARTICULOS FIESTA,17,1,NaN,Cumpleaños / Artículos de Fiesta,NaN


### Productos Interfuerza

Total productos: 73681


,Id,UPC Code,Ubicacion,Item Number,Tipo,Nombre,Proveedor Principal,Marca,Pais de Origen,Punto de ReOrden,...,Matriz Padre,Matriz Hijo,Arancel,Material,UoM InStock,UoM Compra,UoM Venta,Tags,Peso,Detalle
0,PRO101109,7512155423106,BODEGA PRINCIPAL,A404/6917,PRODUCTO,GORRA FBI 7512155423106,OMER SA,GENERICO,NaN,0,...,NO,NO,NaN,NaN,NaN,NaN,NaN,NaN,0.0,SOMBRERO SHERIFF
1,PRO104891,40568000067,NaN,405686/25.95OFERTA5.00,PRODUCTO,VESTIDO LARGO DAMA CHINO,CAPI WORLDWIDE SA,ZZZZZ,NaN,0,...,NO,NO,NaN,NaN,NaN,NaN,NaN,NaN,0.0,VESTIDO LARGO DAMA CHINO
2,PRO107818,7465376497664,NaN,9599/HB2473-18,PRODUCTO,BIRRETE GRADUACION NEGRO-AZUL TELA 7465376497664,OMER SA,GENERICO,NaN,0,...,NO,NO,NaN,NaN,NaN,NaN,NaN,NaN,0.0,VIRRETE GRADUACION TELA
3,PRO107822,6930180607994,NaN,AFD-799,PRODUCTO,NUMEROS LED CHICO AFD-799,SOLARTE DE PANAMA SA,FIESTAS DAISY,NaN,0,...,NO,NO,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NUMEROS LED CHICO AFD-799
4,PRO107852,60000001438,BODEGA PRINCIPAL,601430,PRODUCTO,ARREGLO GRADUACION GLOBOS,CORPORACION DAISY SA,DAISY,NaN,0,...,NO,NO,NaN,NaN,NaN,NaN,NaN,NaN,0.0,ARREGLO GRADUACION GLOBOS


## ANÁLISIS DE POSIBLES DUPLICADOS

In [8]:
# AUDITORÍA DE PRODUCTOS - POSIBLES DUPLICADOS 
FUZZY_THRESHOLD = 90
df_prod = df_odoo.copy() 
def normalize_name(name):
    name = str(name).strip()
    name = unicodedata.normalize('NFKD', name).encode('ASCII', 'ignore').decode('ASCII')
    name = re.sub(r'[^\w\s]', '', name)
    return ' '.join(name.lower().split())

# Normalizar códigos
df_odoo_copy['barcode_norm'] = df_odoo_copy['barcode'].apply(barcode_normalization)
df_odoo_copy['ref_norm'] = df_odoo_copy['default_code'].apply(barcode_normalization)  # ajusta nombre de columna referencia
# Normalizar nombre
df_odoo_copy['name_norm'] = df_odoo_copy['name'].apply(normalize_name)

# PRODUCTOS CON MISMO CÓDIGO DE BARRAS
barcode_valid = df_odoo_copy[df_odoo_copy['barcode_norm'].notna()]

df_same_barcode = (
    barcode_valid.groupby('barcode_norm')
    .agg(nombres=('name', list), cantidad=('name', 'count'))
    .reset_index()
)
df_same_barcode = df_same_barcode[df_same_barcode['cantidad'] > 1].sort_values('cantidad', ascending=False)

display(Markdown(f"### Productos con el mismo código de barras: {len(df_same_barcode)} grupos"))
display(df_same_barcode)
# df_same_barcode.to_csv(processed_path / 'Prod_DupBarcode.csv', index=False)

# PRODUCTOS CON NOMBRES SIMILARES (FUZZY)
names_unique = df_odoo_copy[['name', 'name_norm']].drop_duplicates(subset='name_norm')
norm_list = names_unique['name_norm'].tolist()
orig_list = names_unique['name'].tolist()

# Matriz de similitud completa — vectorizada en C
matrix = process.cdist(norm_list, norm_list, scorer=fuzz.token_sort_ratio)
np.fill_diagonal(matrix, 0)  # ignorar comparación consigo mismo

fuzzy_rows, assigned = [], set()
for i, ni in enumerate(norm_list):
    if ni in assigned:
        continue
    hits = np.where(matrix[i] >= FUZZY_THRESHOLD)[0]
    group = [orig_list[j] for j in hits if norm_list[j] not in assigned]
    if group:
        fuzzy_rows.append({'nombre_original': orig_list[i], 'similares': group, 'cantidad': len(group) + 1})
        assigned.update([ni] + [norm_list[j] for j in hits])

df_fuzzy_prod = pd.DataFrame(fuzzy_rows).sort_values('cantidad', ascending=False)
display(Markdown(f"### Productos con nombres similares: {len(df_fuzzy_prod)} grupos"))
display(df_fuzzy_prod)
# df_fuzzy_prod.to_csv(processed_path / 'Prod_DupFuzzy.csv', index=False)

# CÓDIGO DE BARRAS Y REFERENCIA INVERTIDOS
# Caso: barcode de A == referencia de B y viceversa
df_bc  = df_odoo_copy[df_odoo_copy['barcode_norm'].notna()][['name', 'barcode_norm', 'ref_norm']]
df_ref = df_odoo_copy[df_odoo_copy['ref_norm'].notna()][['name', 'barcode_norm', 'ref_norm']]

# Merge: el barcode de un producto coincide con la referencia de otro
inverted = df_bc.merge(
    df_ref,
    left_on='barcode_norm', right_on='ref_norm',
    suffixes=('_A', '_B')
)
# Excluir el mismo producto comparado consigo mismo
inverted = inverted[inverted['name_A'] != inverted['name_B']]
inverted = inverted[['name_A', 'barcode_norm_A', 'ref_norm_A',
                      'name_B', 'barcode_norm_B', 'ref_norm_B']].drop_duplicates()

display(Markdown(f"### Productos con código de barras y referencia invertidos: {len(inverted)} pares"))
display(inverted.head(20))
# inverted.to_csv(processed_path / 'Prod_DupInvertidos.csv', index=False)

# CAMPOS FALTANTES
campos = {
    'barcode_norm': 'Código de barras',
    'default_code':     'Referencia',
    'categ_id':     'Categoría',
}

faltantes_resumen = []
for col, label in campos.items():
    if col not in df_odoo_copy.columns:
        print(f"⚠️ Columna '{col}' no encontrada")
        continue
    mask = df_odoo_copy[col].isna() | (df_odoo_copy[col].astype(str).str.strip() == '')
    df_faltante = df_odoo_copy[mask][['name'] + [c for c in ['barcode_norm', 'ref_norm', 'categ_id'] if c in df_odoo_copy.columns]]
    display(Markdown(f"### Sin {label}: {mask.sum()} productos"))
    display(df_faltante.head(10))
    # df_faltante.to_csv(processed_path / f'Prod_Sin_{col}.csv', index=False)
    faltantes_resumen.append({'campo': label, 'cantidad': mask.sum()})

# REFERENCIAS IDÉNTICAS
ref_valid = df_odoo_copy[df_odoo_copy['ref_norm'].notna() & (df_odoo_copy['ref_norm'] != '')]

df_same_ref = (
    ref_valid.groupby('ref_norm')
    .agg(nombres=('name', list), cantidad=('name', 'count'))
    .reset_index()
)
df_same_ref = df_same_ref[df_same_ref['cantidad'] > 1].sort_values('cantidad', ascending=False)

display(Markdown(f"### Referencias idénticas: {len(df_same_ref)} grupos"))
display(df_same_ref)
# df_same_ref.to_csv(processed_path / 'Prod_DupReferencia.csv', index=False)

# RESUMEN EJECUTIVO
resumen = pd.DataFrame({
    'Categoría': [
        'Total productos',
        'Grupos: mismo barcode, diferente nombre',
        f'Grupos: nombres similares (fuzzy ≥{FUZZY_THRESHOLD}%)',
        'Pares: código y referencia invertidos',
        'Grupos: referencias idénticas',
        *[f['campo'] + ' faltante' for f in faltantes_resumen],
    ],
    'Cantidad': [
        len(df_odoo_copy),
        len(df_same_barcode),
        len(df_fuzzy_prod),
        len(inverted),
        len(df_same_ref),
        *[f['cantidad'] for f in faltantes_resumen],
    ]
})
display(Markdown("## Resumen del diagnóstico"))
display(resumen)

### Productos con el mismo código de barras: 3 grupos

,barcode_norm,nombres,cantidad
2040,28400019903,"[LAY DORITOS NACHOS CHEESE , LAY FRITOS ORIGIN...",2
2339,38000845529,"[PRINGLES SOUR CREAM & ONION , PRINGLES SOUR C...",2
10583,77780021020,[HILO GUTERMAN 408 ROJO 274YDA GDE CJ/5 077780...,2


### Productos con nombres similares: 910 grupos

,nombre_original,similares,cantidad
659,MI CALIGRAFIA 0,"[MI CALIGRAFIA 1, MI CALIGRAFIA 10, MI CALIGRA...",13
677,NACHO CALIGRAFIA Y ORTOGRAFIA 1,"[NACHO CALIGRAFIA Y ORTOGRAFIA 10, NACHO CALIG...",12
840,SCIENCE INTERNATIONAL PRIMARY STBK LEVEL 1,"[SCIENCE INTERNATIONAL PRIMARY STBK LEVEL 2, S...",12
797,PROGRAMA VIVE 1,"[PROGRAMA VIVE 10, PROGRAMA VIVE 11 , PROGRAMA...",11
710,PANTALON NEGRO S/P T-10 DUREX,"[PANTALON NEGRO S/P T-12 DUREX, PANTALON NEGRO...",10
...,...,...,...
905,VESTIDO TEJIDO ALICIA PEQ.,[VESTIDO TEJIDO GDE ALICIA],2
906,VINCHA BAUTIZO COMUNION 105933,[VINCHA BAUTIZO COMUNION 105934],2
907,VINCHA PLASTICA TRANSP. 1.5 CM,[VINCHA PLASTICA TRANSP. 2CM],2
207,CUADERNO AGUILA DRAGONBALL 200D/R HILO GDE CJ/64,[CUADERNO AGUILA DRAGONBALL 200R/A HILO GDE CJ...,2


### Productos con código de barras y referencia invertidos: 3 pares

...

,name,barcode_norm,ref_norm,categ_id
27,NaN,None,None,NaN
132,NaN,None,None,NaN
468,NaN,None,None,NaN
540,NaN,None,None,NaN
542,NaN,None,None,NaN
601,NaN,None,None,NaN
690,NaN,None,None,NaN
1205,NaN,None,None,NaN
1584,NaN,None,None,NaN
1741,NaN,None,None,NaN


### Referencias idénticas: 70 grupos

,ref_norm,nombres,cantidad
10899,FPL-26C,"[FOLDER MANILA 8.5X11 MORADO METACOLOR, FOLDER...",4
12846,SLW119-5PAX1,"[RESALTADOR ILUMINA FLEX SLW119 ROSADO , RESAL...",4
10288,E1384B,[OLEO MARIES 50ML 106 E1384B ZINC TITANIUM WHI...,4
2278,312975-8,"[GATORADE NARANJA 600ML X8, GATORADE PONCHE DE...",3
3369,505090,"[PRINGLES CHEESE CHEDAR 21G /03808500, PRINGLE...",3
...,...,...,...
11231,HG-6559,"[GRAMA ARTIFICAIL C/FLORES 40X60CM HG-6559, GR...",2
11390,JAV-PA-256,"[ALCOHOL 95% DESNATURAL 480ML, ALCOHOL 95% DES...",2
11392,JAV-PA-260,"[ALCOHOL 95% CON ATOMIZADOR 480ML, ALCOHOL 95%...",2
12369,PESEBRE REF14,"[FONDO DE PESEBRE #14 4867300001436, FONDO DE ...",2


## Resumen del diagnóstico

,Categoría,Cantidad
0,Total productos,14215
1,"Grupos: mismo barcode, diferente nombre",3
2,Grupos: nombres similares (fuzzy ≥90%),910
3,Pares: código y referencia invertidos,3
4,Grupos: referencias idénticas,70
5,Código de barras faltante,248
6,Referencia faltante,325
7,Categoría faltante,189


 ## ANÁLISIS DE PATRONES

In [9]:
# ANÁLISIS DE PATRONES EN NOMBRES DE PRODUCTOS
df_odoo_copy['name_upper'] = df_odoo_copy['name'].str.upper().str.strip()

# Columnas de referencia para sección 4
REF_COL      = 'default_code'
BARCODE_COL  = 'barcode'
BRAND_COL    = 'x_studio_many2many_field_37q_1irl4uc58'

results = {}  # acumula conteos para resumen

def tag(df, mask, label):
    """Registra conteo y retorna subconjunto."""
    results[label] = mask.sum()
    return df[mask][['name']].copy()

# SECCIÓN 1 — ABREVIACIONES
display(Markdown("## Sección 1 — Abreviaciones en el nombre"))

abreviaciones = {
    'Paquete (PAQ/PQ)':          r'\b(?:PAQ|PQ)\b',
    'Piezas (PZ/PZS/PZA/PCS…)':  r'\b(?:PZ|PZS|PZAS|PZA|PCS|PIEZAS)\b',
    'Medidas':                    r'\b(?:CM|MM|YDS?|YDAS?|PULG|MTS?|ML|GR)\b',
    'Transparente (TRANSP…)':     r'\bTRANSP\w*\b',
    'Juego (JGO…)':               r'\bJGO\w*\b',
    'Surtido (SURT…)':            r'\bSURTI?D?O?\b',
    'Plástico (PLAST)':           r'\bPLAST\b',
    'Páginas (PAG)':              r'\bPAG\b',
    'Unidades (UNDS)':            r'\bUNDS\b',
}

for label, pattern in abreviaciones.items():
    mask = df_odoo_copy['name_upper'].str.contains(pattern, regex=True, na=False)
    sub  = tag(df_odoo_copy, mask, label)
    display(Markdown(f"### {label}: {mask.sum()} productos"))
    display(sub.head(10))

# SECCIÓN 2 — CANTIDADES Y MEDIDAS NUMÉRICAS
display(Markdown("## Sección 2 — Cantidades y medidas numéricas"))

cantidades = {
    'Multiplicador X (X8, X 12…)':       r'\bX\s*\d+\b',
    'Medidas numéricas (2.5X6, 40CMX…)': r'\d+[\.,]?\d*\s*[A-Z]*\s*X\s*\d+[\.,]?\d*',
    'Hojas H (50H, 400 H…)':             r'\b\d+\s*H\b',
    'Cantidad por caja (CJ/N, CAJAX…)':  r'\b(?:CJA?X?|C|CAJA)\s*[/X]?\s*\d+\b',
}

for label, pattern in cantidades.items():
    mask = df_odoo_copy['name_upper'].str.contains(pattern, regex=True, na=False)
    sub  = tag(df_odoo_copy, mask, label)
    display(Markdown(f"### {label}: {mask.sum()} productos"))
    display(sub.head(10))

# SECCIÓN 3 — VALORES DE OTROS CAMPOS EN EL NOMBRE
display(Markdown("## Sección 3 — Otros campos presentes en el nombre"))

# Referencia en nombre
if REF_COL in df_odoo_copy.columns:
    df_ref_check = df_odoo_copy[df_odoo_copy[REF_COL].notna()].copy()
    df_ref_check['ref_upper'] = df_ref_check[REF_COL].astype(str).str.upper().str.strip()
    mask_ref = df_ref_check.apply(
        lambda r: bool(r['ref_upper']) and r['ref_upper'] in r['name_upper'], axis=1
    )
    sub_ref = df_ref_check[mask_ref][['name', REF_COL]]
    results['Referencia en nombre'] = mask_ref.sum()
    display(Markdown(f"### Referencia en nombre: {mask_ref.sum()} productos"))
    display(sub_ref.head(10))
    #sub_ref.to_csv(processed_path / 'Prod_RefEnNombre.csv', index=False)

# Código de barras en nombre
if BARCODE_COL in df_odoo_copy.columns:
    df_bc_check = df_odoo_copy[df_odoo_copy['barcode_norm'].notna()].copy()
    mask_bc = df_bc_check.apply(
        lambda r: str(r['barcode_norm']) in r['name_upper'], axis=1
    )
    sub_bc = df_bc_check[mask_bc][['name', BARCODE_COL]]
    results['Barcode en nombre'] = mask_bc.sum()
    display(Markdown(f"### Código de barras en nombre: {mask_bc.sum()} productos"))
    display(sub_bc.head(10))
    #sub_bc.to_csv(processed_path / 'Prod_BarcodeEnNombre.csv', index=False)

# Marca en nombre
if BRAND_COL in df_odoo_copy.columns:
    df_brand_check = df_odoo_copy[df_odoo_copy[BRAND_COL].notna()].copy()
    df_brand_check['brand_upper'] = df_brand_check[BRAND_COL].astype(str).str.upper().str.strip()
    mask_brand = df_brand_check.apply(
        lambda r: bool(r['brand_upper']) and r['brand_upper'] in r['name_upper'], axis=1
    )
    sub_brand = df_brand_check[mask_brand][['name', BRAND_COL]]
    results['Marca en nombre'] = mask_brand.sum()
    display(Markdown(f"### Marca en nombre: {mask_brand.sum()} productos"))
    display(sub_brand.head(10))
    #sub_brand.to_csv(processed_path / 'Prod_MarcaEnNombre.csv', index=False)

# SECCIÓN 4 — CARACTERES ESPECIALES
display(Markdown("## Sección 4 — Caracteres especiales en el nombre"))
especiales = {
    'Punto  (.)': r'\.',
    'Barra  (/)': r'/',
    'Guion  (-)': r'-',
}

for label, pattern in especiales.items():
    mask = df_odoo_copy['name'].str.contains(pattern, regex=True, na=False)
    sub  = tag(df_odoo_copy, mask, label)
    display(Markdown(f"### {label}: {mask.sum()} productos"))
    display(sub.head(10))

# Cualquier carácter especial combinado
mask_any = df_odoo_copy['name'].str.contains(r'[./\-]', regex=True, na=False)
results['Algún carácter especial'] = mask_any.sum()
df_odoo_copy['tiene_especial'] = mask_any
#df_odoo_copy[mask_any][['name']].to_csv(processed_path / 'Prod_CaracteresEspeciales.csv', index=False)

# RESUMEN
resumen = pd.DataFrame({
    'Sección':   ['Abreviaciones'] * len(abreviaciones) +
                 ['Cantidades']    * len(cantidades) +
                 ['Otros campos']  * 3 +
                 ['Caract. especiales'] * (len(especiales) + 1),
    'Patrón':    list(abreviaciones) + list(cantidades) +
                 ['Referencia en nombre', 'Barcode en nombre', 'Marca en nombre'] +
                 list(especiales) + ['Algún carácter especial'],
    'Productos': [results.get(k, 'N/A') for k in
                  list(abreviaciones) + list(cantidades) +
                  ['Referencia en nombre', 'Barcode en nombre', 'Marca en nombre'] +
                  list(especiales) + ['Algún carácter especial']]
})
display(Markdown("## Resumen ejecutivo — Patrones en nombres"))
display(resumen)

## Sección 1 — Abreviaciones en el nombre

### Paquete (PAQ/PQ): 272 productos

,name
265,AGUJA FACIL ENHEBRAR PQ
274,"AGUJA S/P PAQ.5 #18,20,22,24 CANEVA PONY"
359,ALFA CALCOMANIA SURTIDA PQ.5
510,ANIMAL SURT PAQ
521,ANIMALES SALVAJES PAQ WEINIDA
626,APLICACION INFANTIL TULL LENT. MARIPOSA PQ.5
725,ARGOLLA LLAVERO PAQ
1019,BARAJAS PAQ TUN HUANG
1104,BEBE ACOST.X6 PAQ
1108,BEBE SENTADO/GANZO PQ.6PZA


### Piezas (PZ/PZS/PZA/PCS…): 302 productos

,name
28,FOAMI LISO 8.5X11 SURT. 10 PCS
29,FOAMI LISO 8.5X11 SURT. 10 PCS
30,FOAMI LISO 8.5X11 SURT. 10 PCS
505,ANILLOS 12 PCS PRETTY GIRL
854,ARTESCO BORRADOR BLANCO X 2 PZA GDE
1122,BENY MALVAS MAXY BOMBOM 25 PZA/650G
1260,BLOCK DIDACTICO 83 PCS LE JOY BABY
1290,BLOQUES DE CONSTRUCCION FOMI 20 PIEZAS 7453010...
1291,BLOQUES DIDACTICO MADERA CUBO X125 PIEZAS POINTER
1292,BLOQUES DIDACTICOS X50 PZAS


...

,name
8,CADENA STRASS 3 LINEAS C/PIEDRA X YDA
13,CINTA SATIN 1/4 (6MM) X 109YDS R-5.45
34,SET COLETAS P/CABELLO
75,ABREMENTE 2-1 ARTE/DEPORTE
80,ACCESORIO P/ CABELLO
81,ACCESORIO P/MEJORAR BRILLO MEDIUM FOR OIL COLO...
88,ACOPLE TRAI 47-940 1-7/8X2
93,ACORDEON KRACH F/C 13 DIVISION STUDMARK
110,ACRILICA PROFESSIONAL 204/100ML AMARILLO CADMIO
118,ACRILICO PROFESIONAL 205/100ML SOMBRA TOSTADA


### Guion  (-): 2764 productos

,name
13,CINTA SATIN 1/4 (6MM) X 109YDS R-5.45
15,DESMALEZADORA MULTI-HERRAMIENTA PARA PODAR
20,PAPEL KRAFT 36X32 150YDS R-75.00
25,VASO 10OZ X 30 ECO-AMIGABLE 764246005624
33,LAMINADO PARA PLASTIFICAR 9X14.5 C-100 229mmx...
37,20 AÑOS DESPUES - Rose Marie Tapia
45,A MIS NINOS CON AMOR - CARMEN DALILA SANTAMARIA
46,A MIS NINOS CON AMOR-TALLERES DE RECUPERACION ...
74,ABRAZADERA 19MM-38 COVO
75,ABREMENTE 2-1 ARTE/DEPORTE


## Resumen ejecutivo — Patrones en nombres

,Sección,Patrón,Productos
0,Abreviaciones,Paquete (PAQ/PQ),272
1,Abreviaciones,Piezas (PZ/PZS/PZA/PCS…),302
2,Abreviaciones,Medidas,489
3,Abreviaciones,Transparente (TRANSP…),139
4,Abreviaciones,Juego (JGO…),15
5,Abreviaciones,Surtido (SURT…),408
6,Abreviaciones,Plástico (PLAST),49
7,Abreviaciones,Páginas (PAG),27
8,Abreviaciones,Unidades (UNDS),10
9,Cantidades,"Multiplicador X (X8, X 12…)",798
